# Exercise 07: Change the feature structure and prediction horizon

Tutorial 07 compared ANN and LSTM with one fixed feature design. This exercise uses the same PeMSD4 detector but changes choices that a modeller must make: hidden size, input variables, history length and prediction horizon.

Use training data to fit parameters, validation data to select each configuration, and test data once for the final comparison. Do not use test results to revise a model.


## Open this notebook

- **Colab.** Open the notebook, then run the setup cell below before any data-loading cell. It clones the public course repository and changes into `07 Recurrent neural networks`, so relative data paths work.
- **On your own computer.** Open the notebook from its lesson folder so the data files sit beside it.


In [ ]:
from pathlib import Path
import os

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    print("Running in Google Colab")
    REPO_URL = "https://github.com/hku-kejintao/AI-Code.git"
    LESSON_DIR = "07 Recurrent neural networks"
    base = Path("/content")
    repo = base / "AI-Code"
    os.chdir(base)
    if not repo.exists():
        !git clone {REPO_URL}
    lesson_path = repo / LESSON_DIR
    if not lesson_path.exists():
        raise FileNotFoundError(f"Lesson folder not found: {lesson_path}")
    os.chdir(lesson_path)
else:
    print("Running on your own computer")

print("Current folder:", os.getcwd())


### Check the packages

The cell should print version numbers and the word `ready`. NumPy holds arrays, pandas holds tables, Matplotlib draws figures, and scikit-learn provides preprocessing, models, and metrics. PyTorch builds the neural networks used in this lesson.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)
print("ready")


### Install, only if the check failed

Skip this cell when the check printed `ready`. After installation finishes, restart the kernel and run the cells above again.


In [ ]:
%pip install numpy pandas matplotlib scikit-learn torch


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(2)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

In [ ]:
def make_one_step_windows(array, lookback, forecast_horizon):
    input_windows = []
    flow_targets = []
    target_positions = []

    first_target = lookback + forecast_horizon - 1
    for target_position in range(first_target, len(array)):
        input_end = target_position - forecast_horizon
        input_start = input_end - lookback + 1
        input_windows.append(array[input_start : input_end + 1])
        flow_targets.append(array[target_position, 0])
        target_positions.append(target_position)

    X = np.stack(input_windows).astype("float32")
    y = np.asarray(flow_targets, dtype="float32").reshape(-1, 1)
    return X, y, np.asarray(target_positions)


class TrafficANN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x):
        # The exercise must supply x as (N, T × D), not (N, T, D).
        return self.network(x)


class TrafficLSTM(nn.Module):
    def __init__(self, n_features, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        hidden_sequence, _ = self.lstm(x)
        return self.output(hidden_sequence[:, -1, :])

def fit_model(model, train_loader, valid_loader, epochs=8, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    history = {"train MAE": [], "valid MAE": []}
    best_state = None
    best_loss = float("inf")
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_absolute_error = 0
        train_targets = 0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            prediction = model(xb)
            loss = loss_fn(prediction, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(xb)
            train_absolute_error += torch.abs((prediction - yb) * flow_std).sum().item()
            train_targets += yb.numel()

        model.eval()
        valid_loss = 0
        valid_absolute_error = 0
        valid_targets = 0
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                prediction = model(xb)
                valid_loss += loss_fn(prediction, yb).item() * len(xb)
                valid_absolute_error += torch.abs((prediction - yb) * flow_std).sum().item()
                valid_targets += yb.numel()
        train_loss /= len(train_loader.dataset)
        valid_loss /= len(valid_loader.dataset)
        train_mae = train_absolute_error / train_targets
        valid_mae = valid_absolute_error / valid_targets
        history["train MAE"].append(train_mae)
        history["valid MAE"].append(valid_mae)
        if valid_mae < best_loss:
            best_loss = valid_mae
            best_state = copy.deepcopy(model.state_dict())
        print(
            f"epoch {epoch+1:02d} | "
            f"train MAE {train_mae:.2f} | "
            f"validation MAE {valid_mae:.2f} vehicles/5 min"
        )
    model.load_state_dict(best_state)
    return history

def predict(model, X, batch_size=256):
    model.eval()
    outputs = []
    loader = DataLoader(
        TensorDataset(torch.tensor(X)),
        batch_size=batch_size,
    )
    with torch.no_grad():
        for (xb,) in loader:
            outputs.append(model(xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(outputs)

In [ ]:
traffic = pd.read_csv(
    "pems04_single_sensor.csv.gz",
    parse_dates=["time"],
).set_index("time")
traffic = traffic.iloc[:12000].copy()
train_end = 7500
valid_end = 9500
values = traffic.to_numpy(dtype="float32")
feature_mean = values[:train_end].mean(axis=0)
feature_std = values[:train_end].std(axis=0)
scaled_values = (values - feature_mean) / feature_std
flow_std = float(feature_std[0])
print("Raw traffic table:", traffic.shape, "= (five-minute time steps, variables)")
print("Preview in original units: flow [vehicles/5 min], occupancy [fraction], and speed.")
display(traffic.head())
print("Scaling is used only for ANN/LSTM fitting; reported errors are converted back to flow units.")

## Question 1 — Verify $N$, $T$, $D$ and the baseline

Use $T=12$, $D=3$ and a 12-interval forecast horizon. Create the six arrays `X_train`, `y_train`, `X_valid`, `y_valid`, `X_test`, and `y_test`, then print every shape and explain the meanings and values of $N$, $T$ and $D$. On validation data, calculate persistence MAE and RMSE in the original flow unit. Do not inspect test performance yet.


In [ ]:
# TODO: print X/y shapes and identify N, T and D.
# TODO: calculate persistence MAE in original flow units.


## Question 2 — Tune ANN and LSTM capacity fairly

Using all three variables and the same windows, explicitly reshape ANN inputs to $(N,T\times D)$ while retaining LSTM inputs as $(N,T,D)$. Print both shapes, then compare hidden sizes 32 and 64 separately for ANN and LSTM. Select one hidden size per model using validation MAE. Plot a grouped validation comparison and freeze both selected models. Explain why this comparison changes the architecture but not the information supplied to it.


In [ ]:
# TODO: flatten ANN input to (N, T*D), keep LSTM input as (N, T, D).
# TODO: tune ANN width and LSTM hidden size on validation MAE.


## Question 3 — Test whether all three variables help

For the selected LSTM capacity, compare two feature structures on validation data: (a) past flow only, giving $(N,T,1)$, and (b) flow, occupancy and speed, giving $(N,T,3)$. Keep the split, target and training budget fixed. State what this ablation can and cannot tell us.


In [ ]:
# TODO: compare flow-only and flow+occupancy+speed LSTM inputs on validation data.


## Question 4 — Change the history length

Using the validation-best feature structure, compare lookbacks of 6, 12 and 24 intervals for LSTM. Plot validation MAE against history length in minutes, choose the best $T$, and freeze the resulting LSTM. Longer history is not automatically better.


In [ ]:
# TODO: compare the supplied lookback lengths using validation MAE.
# Keep the forecast horizon fixed while changing T.


## Question 5 — Final test comparison and multi-step extension

First, evaluate persistence, the frozen ANN and the frozen LSTM on test data once. Report MAE and RMSE and plot two test days.

Then change the target from $(N,1)$ to $(N,12,1)$ so that one model predicts the next hour of flow. Modify the LSTM output layer to return 12 steps, train it without using test data for decisions, and finally plot test MAE against forecast horizon (5, 10, ..., 60 minutes). Compare it with a persistence sequence that repeats the final observed flow 12 times.


In [ ]:
# TODO: evaluate frozen persistence, ANN and LSTM models on test once.
# TODO: train direct LSTM outputs for 1, 3 and 5 future steps and plot horizon-wise MAE.
